# Speaker identification & diarization — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(z):
    z=np.asarray(z,dtype=float); e=np.exp(z-z.max()); return e/e.sum()
def cosine(a,b):
    a=np.asarray(a,dtype=float); b=np.asarray(b,dtype=float); return float(a@b/(np.linalg.norm(a)*np.linalg.norm(b)))
def hz_to_mel(f): return 2595*np.log10(1+np.asarray(f)/700)
def mel_to_hz(m): return 700*(10**(np.asarray(m)/2595)-1)
def stft_mag(x,N=256,H=64):
    w=np.hanning(N); frames=[]
    for s in range(0,len(x)-N+1,H): frames.append(np.abs(np.fft.rfft(x[s:s+N]*w)))
    return np.array(frames).T
print('setup ok')

## Voice embeddings and who-spoke-when timelines
Speaker systems use embedding geometry and timeline segmentation. These examples show cosine scoring, thresholds, clustering, overlap, and a diarization track.

In [ ]:
# 1) cosine similarity identifies the enrolled speaker
q=np.array([1.,2.]); A=np.array([1.2,2.1]); B=np.array([-1.,0.]); scores=[cosine(q,A),cosine(q,B)]
plt.figure(figsize=(6,3)); plt.bar(['A','B'],scores); plt.axhline(0.75,c='k',ls='--'); plt.ylim(-1,1); plt.title('query matches speaker A')
assert abs(scores[0]-0.998)<0.001 and abs(scores[1]+0.447)<0.001

In [ ]:
# 2) thresholding trades accepts and rejects
scores=np.array([0.95,0.82,0.62,0.1,-0.2]); labels=np.array([1,1,1,0,0]); th=.75; pred=scores>=th; tp=np.sum(pred & (labels==1)); fp=np.sum(pred & (labels==0)); fn=np.sum((~pred) & (labels==1))
plt.figure(figsize=(6,3)); plt.scatter(scores,labels); plt.axvline(th,c='r',ls='--'); plt.xlabel('cosine score'); plt.ylabel('same speaker'); plt.title(f'TP={tp}, FP={fp}, FN={fn}')
assert (tp,fp,fn)==(2,0,1)

In [ ]:
# 3) simple clustering recovers two speaker groups
rng=np.random.default_rng(1); emb=np.r_[rng.normal([1,0],.08,(8,2)),rng.normal([-1,0],.08,(8,2))]; assign=(emb[:,0]<0).astype(int)
plt.figure(figsize=(4,4)); plt.scatter(emb[:,0],emb[:,1],c=assign); plt.axvline(0,c='k',ls='--'); plt.title('embedding clusters')
assert assign[:8].sum()==0 and assign[8:].sum()==8

In [ ]:
# 4) overlap matters in diarization timelines
segments=[(0,2,'A'),(1,3,'B'),(3,4,'A')]; overlap=sum(max(0,min(e1,e2)-max(s1,s2)) for i,(s1,e1,a) in enumerate(segments) for s2,e2,b in segments[i+1:] if a!=b); frac=overlap/4
plt.figure(figsize=(6,2));
for s,e,a in segments: plt.plot([s,e],[1 if a=='A' else 0]*2,lw=8,label=a)
plt.yticks([0,1],['B','A']); plt.xlim(0,4); plt.title(f'overlap fraction={frac:.2f}')
assert overlap==1 and abs(frac-0.25)<1e-9

In [ ]:
# 5) end-to-end toy diarization from frame embeddings
frames=np.arange(8); emb=np.array([[1,0],[.9,.1],[1.1,0],[-1,0],[-.9,.1],[-1.1,0],[1,0],[.95,.05]]); spk=(emb[:,0]<0).astype(int)
plt.figure(figsize=(6,2)); plt.imshow(spk[None,:],aspect='auto',cmap='coolwarm'); plt.yticks([]); plt.xticks(frames); plt.title('who spoke when: A A A B B B A A')
assert spk.tolist()==[0,0,0,1,1,1,0,0]